# Notebook 03: Compute Gene Lengths

Calculate exonic union length per gene from GENCODE v19 GTF.

**Method:** For each gene, merge overlapping exons (pyranges `merge(by="gene_id")`),
then sum the non-overlapping widths. This is equivalent to R's
`GenomicRanges::reduce() |> sum(width())`.

**Key details:**
- pyranges auto-converts GTF 1-based to 0-based half-open; `End - Start` = correct width
- `merge(by="gene_id")` ensures only exons from the same gene merge together
- Cache parsed exons as parquet for reuse in notebook 05
- Handle 17 unmatched genes (rescue 15 by symbol, drop 2)

In [1]:
import pandas as pd
import numpy as np
import pyranges as pr
from pathlib import Path

RAW_DIR = Path("../data/raw")
CACHE_DIR = Path("../data/cache")
PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../reference")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Parse GENCODE v19 GTF

Extract exon features, strip version suffixes, filter to protein-coding genes.

In [2]:
cache_path = CACHE_DIR / "gencode_v19_exons.parquet"

if cache_path.exists():
    print("Loading cached exon data from parquet...")
    exon_df = pd.read_parquet(cache_path)
    print(f"Loaded {len(exon_df):,} exon records")
else:
    print("Parsing GENCODE v19 GTF (this takes 1-2 minutes)...")
    gtf = pr.read_gtf(str(RAW_DIR / "gencode.v19.annotation.gtf.gz"))
    print(f"Parsed {len(gtf):,} total GTF entries")
    
    # Filter to exon features
    exons = gtf[gtf.Feature == "exon"]
    exon_df = exons.df.copy()
    print(f"Exon entries: {len(exon_df):,}")
    
    # Strip version suffixes from gene_id
    exon_df["gene_id"] = exon_df["gene_id"].str.replace(r"\.\d+$", "", regex=True)
    
    # Filter to protein-coding
    exon_df = exon_df[exon_df["gene_type"] == "protein_coding"].copy()
    print(f"Protein-coding exon entries: {len(exon_df):,}")
    
    # Drop unnecessary columns to save memory
    keep_cols = ["Chromosome", "Start", "End", "Strand", "gene_id", "gene_name", "gene_type"]
    keep_cols = [c for c in keep_cols if c in exon_df.columns]
    exon_df = exon_df[keep_cols].copy()
    
    # Cache for reuse
    exon_df.to_parquet(cache_path, index=False)
    print(f"Cached {len(exon_df):,} exon records to {cache_path}")

print(f"\nUnique protein-coding genes: {exon_df['gene_id'].nunique():,}")

Loading cached exon data from parquet...


Loaded 1,070,777 exon records

Unique protein-coding genes: 20,345


## 2. Merge Overlapping Exons and Compute Gene Lengths

`merge(by="gene_id")` merges overlapping intervals within each gene separately.
Strand-awareness is on by default (correct for same-gene exons).

In [3]:
# Reconstruct PyRanges from the cleaned DataFrame
exons_pr = pr.PyRanges(exon_df)

# Merge overlapping exons PER GENE (equivalent to R's reduce())
print("Merging overlapping exons per gene...")
merged = exons_pr.merge(by="gene_id")
merged_df = merged.df.copy()
print(f"Merged intervals: {len(merged_df):,} (from {len(exon_df):,} exon records)")

# Calculate width of each non-overlapping interval
# pyranges uses 0-based half-open coordinates: width = End - Start
merged_df["width"] = merged_df["End"] - merged_df["Start"]

# Sum widths per gene
gene_lengths = (
    merged_df
    .groupby("gene_id")["width"]
    .sum()
    .reset_index()
    .rename(columns={"gene_id": "ensembl_id", "width": "gene_length_bp"})
)

print(f"\nComputed exonic union lengths for {len(gene_lengths):,} protein-coding genes")
print(f"\nGene length statistics:")
print(gene_lengths["gene_length_bp"].describe())

Merging overlapping exons per gene...


Merged intervals: 239,950 (from 1,070,777 exon records)

Computed exonic union lengths for 20,345 protein-coding genes

Gene length statistics:
count     20345.000000
mean       4680.652249
std        3513.290609
min          21.000000
25%        2239.000000
50%        3909.000000
75%        6284.000000
max      118327.000000
Name: gene_length_bp, dtype: float64


## 3. Handle Unmatched Genes (Rescue by Symbol)

Some Hubstenberger genes may not match GENCODE by Ensembl ID. Jason rescued 15 of 17
by gene symbol lookup; 2 (HEATR9, BAHCC1) were dropped.

In [4]:
# Load our protein-coding gene list from notebook 02
hub_pc = pd.read_csv(PROCESSED_DIR / "02_protein_coding_genes.csv")

# Join gene lengths
hub_with_len = hub_pc.merge(gene_lengths, on="ensembl_id", how="left")

missing_len = hub_with_len[hub_with_len["gene_length_bp"].isna()]
print(f"Genes with matched length: {hub_with_len['gene_length_bp'].notna().sum():,}")
print(f"Genes missing length: {len(missing_len):,}")

if len(missing_len) > 0:
    print(f"\nGenes missing lengths:")
    # Try to identify gene names for missing genes
    name_cols = [c for c in missing_len.columns if 'name' in c.lower() or 'symbol' in c.lower() or 'gene' in c.lower()]
    if name_cols:
        for _, row in missing_len.iterrows():
            names = [str(row[c]) for c in name_cols[:2] if pd.notna(row[c])]
            print(f"  {row['ensembl_id']} ({', '.join(names)})")
    else:
        for eid in missing_len["ensembl_id"]:
            print(f"  {eid}")

Genes with matched length: 15,148
Genes missing length: 317

Genes missing lengths:
  ENSG00000277775 (ENSG00000277775, HIST1H3F)
  ENSG00000280071 (ENSG00000280071, CH507-9B2.3)
  ENSG00000277157 (ENSG00000277157, HIST1H4D)
  ENSG00000275379 (ENSG00000275379, HIST1H3I)
  ENSG00000249156 (ENSG00000249156, RP11-432M8.9)
  ENSG00000275714 (ENSG00000275714, HIST1H3A)
  ENSG00000276966 (ENSG00000276966, HIST1H4E)
  ENSG00000273802 (ENSG00000273802, HIST1H2BG)
  ENSG00000276368 (ENSG00000276368, HIST1H2AJ)
  ENSG00000277791 (ENSG00000277791, PSMB3)
  ENSG00000282757 (ENSG00000282757, DUXB)
  ENSG00000276903 (ENSG00000276903, HIST1H2AL)
  ENSG00000274641 (ENSG00000274641, HIST1H2BO)
  ENSG00000277224 (ENSG00000277224, HIST1H2BF)
  ENSG00000278463 (ENSG00000278463, HIST1H2AB)
  ENSG00000274997 (ENSG00000274997, HIST1H2AH)
  ENSG00000278705 (ENSG00000278705, HIST1H4B)
  ENSG00000160963 (ENSG00000160963, COL26A1)
  ENSG00000277075 (ENSG00000277075, HIST1H2AE)
  ENSG00000273703 (ENSG00000273703,

In [5]:
# Build a gene_name -> (ensembl_id, gene_length_bp) mapping from GENCODE
# for rescue by symbol
gencode_name_to_length = {}
name_df = exon_df[["gene_id", "gene_name"]].drop_duplicates()
for _, row in name_df.iterrows():
    gname = row["gene_name"]
    gid = row["gene_id"]
    if gname not in gencode_name_to_length and gid in gene_lengths["ensembl_id"].values:
        length = gene_lengths[gene_lengths["ensembl_id"] == gid]["gene_length_bp"].values[0]
        gencode_name_to_length[gname] = (gid, length)

# Attempt rescue for missing genes
rescue_records = []
unrescued = []

# Find the gene name column in hub data
gene_name_col = None
for c in hub_with_len.columns:
    if 'associated gene name' in c.lower() or c == 'Associated Gene Name':
        gene_name_col = c
        break
    if 'gene_name' in c.lower() or 'symbol' in c.lower():
        gene_name_col = c
        break

if gene_name_col and len(missing_len) > 0:
    print(f"Using column '{gene_name_col}' for symbol-based rescue\n")
    for idx, row in missing_len.iterrows():
        symbol = row[gene_name_col] if pd.notna(row.get(gene_name_col)) else None
        if symbol and symbol in gencode_name_to_length:
            rescue_eid, rescue_len = gencode_name_to_length[symbol]
            hub_with_len.loc[idx, "gene_length_bp"] = rescue_len
            hub_with_len.loc[idx, "rescue_ensembl_gene_id"] = rescue_eid
            rescue_records.append((row["ensembl_id"], symbol, rescue_eid, rescue_len))
            print(f"  RESCUED: {symbol} ({row['ensembl_id']} -> {rescue_eid}, length={rescue_len:,})")
        else:
            unrescued.append((row["ensembl_id"], symbol))
            print(f"  DROPPED: {symbol} ({row['ensembl_id']}) — no GENCODE match")

print(f"\nRescued: {len(rescue_records)}")
print(f"Dropped: {len(unrescued)}")

Using column 'Associated Gene Name' for symbol-based rescue

  RESCUED: HIST1H3F (ENSG00000277775 -> ENSG00000256316, length=466)
  DROPPED: CH507-9B2.3 (ENSG00000280071) — no GENCODE match
  RESCUED: HIST1H4D (ENSG00000277157 -> ENSG00000188987, length=367)
  RESCUED: HIST1H3I (ENSG00000275379 -> ENSG00000182572, length=477)
  DROPPED: RP11-432M8.9 (ENSG00000249156) — no GENCODE match
  RESCUED: HIST1H3A (ENSG00000275714 -> ENSG00000198366, length=469)
  RESCUED: HIST1H4E (ENSG00000276966 -> ENSG00000198518, length=1,409)
  RESCUED: HIST1H2BG (ENSG00000273802 -> ENSG00000187990, length=445)
  RESCUED: HIST1H2AJ (ENSG00000276368 -> ENSG00000182611, length=496)
  RESCUED: PSMB3 (ENSG00000277791 -> ENSG00000108294, length=1,179)
  DROPPED: DUXB (ENSG00000282757) — no GENCODE match
  RESCUED: HIST1H2AL (ENSG00000276903 -> ENSG00000198374, length=573)
  RESCUED: HIST1H2BO (ENSG00000274641 -> ENSG00000196331, length=467)
  RESCUED: HIST1H2BF (ENSG00000277224 -> ENSG00000197846, length=1,195

  RESCUED: ANXA8L1 (ENSG00000264230 -> ENSG00000150165, length=2,127)
  DROPPED: SCART1 (ENSG00000214279) — no GENCODE match
  DROPPED: CTC-441N14.4 (ENSG00000250803) — no GENCODE match
  DROPPED: RP11-722G7.1 (ENSG00000269226) — no GENCODE match
  DROPPED: RP11-345F18.1 (ENSG00000250821) — no GENCODE match
  DROPPED: ERVH48-1 (ENSG00000233056) — no GENCODE match
  DROPPED: RP11-402G3.5 (ENSG00000230054) — no GENCODE match
  DROPPED: RAB7B (ENSG00000276600) — no GENCODE match
  DROPPED: LINC00371 (ENSG00000226792) — no GENCODE match
  DROPPED: SPDYE11 (ENSG00000275976) — no GENCODE match
  DROPPED: LINC01314 (ENSG00000259417) — no GENCODE match
  DROPPED: LINC00854 (ENSG00000236383) — no GENCODE match
  RESCUED: PCDH20 (ENSG00000280165 -> ENSG00000197991, length=7,053)
  RESCUED: PAGR1 (ENSG00000280789 -> ENSG00000185928, length=2,165)
  RESCUED: HIST1H2BE (ENSG00000274290 -> ENSG00000197697, length=497)
  RESCUED: GJA5 (ENSG00000265107 -> ENSG00000143140, length=3,256)
  DROPPED: AC24

In [6]:
# Drop genes that could not be rescued (no gene length available)
before = len(hub_with_len)
hub_with_len = hub_with_len[hub_with_len["gene_length_bp"].notna()].copy()
hub_with_len["gene_length_bp"] = hub_with_len["gene_length_bp"].astype(int)
after = len(hub_with_len)

print(f"Before dropping: {before:,}")
print(f"After dropping unmatched: {after:,}")
print(f"Dropped: {before - after}")
print(f"\nFinal protein-coding gene count with lengths: {after:,}")

Before dropping: 15,465
After dropping unmatched: 15,281
Dropped: 184

Final protein-coding gene count with lengths: 15,281


## 4. Validation Against Jason's Reference

Compare our computed gene lengths against the `gene_length_bp` column in Jason's
12,544-gene reference CSV. Expect exact integer matches since both derive from GENCODE v19.

In [7]:
from scipy import stats

# Load reference
ref = pd.read_csv(REFERENCE_DIR / "12544_Hub_CAGE_MERGE_with_CapIndex.csv")

# Join our lengths with reference on ensembl_id
comparison = ref[["ensembl_id", "gene_length_bp"]].merge(
    gene_lengths,
    on="ensembl_id",
    how="inner",
    suffixes=("_ref", "_ours")
)

print(f"Genes compared: {len(comparison):,} / {len(ref):,} reference genes")

# Pearson correlation
r, p = stats.pearsonr(comparison["gene_length_bp_ref"], comparison["gene_length_bp_ours"])
print(f"\nPearson r: {r:.10f}")

# Exact matches
exact = (comparison["gene_length_bp_ref"] == comparison["gene_length_bp_ours"]).sum()
print(f"Exact matches: {exact:,} / {len(comparison):,} ({exact/len(comparison)*100:.1f}%)")

# Max absolute difference
diff = (comparison["gene_length_bp_ref"] - comparison["gene_length_bp_ours"]).abs()
print(f"Max absolute diff: {diff.max()}")
print(f"Mean absolute diff: {diff.mean():.2f}")

# Show worst discrepancies if any
if diff.max() > 0:
    worst = comparison.loc[diff.nlargest(10).index]
    print(f"\nTop 10 discrepancies:")
    for _, row in worst.iterrows():
        gene = ref[ref["ensembl_id"] == row["ensembl_id"]]["Associated Gene Name"].values[0]
        print(f"  {gene} ({row['ensembl_id']}): ref={row['gene_length_bp_ref']:,} ours={row['gene_length_bp_ours']:,} diff={abs(row['gene_length_bp_ref']-row['gene_length_bp_ours']):,}")

Genes compared: 12,506 / 12,544 reference genes

Pearson r: 1.0000000000
Exact matches: 12,506 / 12,506 (100.0%)
Max absolute diff: 0
Mean absolute diff: 0.00


In [8]:
# Check the 11 rescued genes visible in reference
ref_rescued = ref[ref["rescue_ensembl_gene_id"].notna()]
print(f"Rescued genes in reference: {len(ref_rescued)}")
print(f"\nRescued gene comparison:")
for _, row in ref_rescued.iterrows():
    gene = row["Associated Gene Name"]
    ref_len = row["gene_length_bp"]
    our_match = gene_lengths[gene_lengths["ensembl_id"] == row["ensembl_id"]]
    if len(our_match) > 0:
        our_len = our_match["gene_length_bp"].values[0]
        match = "MATCH" if ref_len == our_len else f"DIFF (ours={our_len:,})"
    else:
        # Check if rescued via different Ensembl ID
        rescue_match = gene_lengths[gene_lengths["ensembl_id"] == row["rescue_ensembl_gene_id"]]
        if len(rescue_match) > 0:
            our_len = rescue_match["gene_length_bp"].values[0]
            match = f"RESCUED (ours={our_len:,})" if ref_len == our_len else f"RESCUED DIFF (ours={our_len:,})"
        else:
            match = "NOT FOUND"
    print(f"  {gene}: ref={ref_len:,} — {match}")

Rescued genes in reference: 11

Rescued gene comparison:
  IKBKE: ref=3,568 — RESCUED (ours=3,568)
  EPPK1: ref=7,997 — RESCUED (ours=7,997)
  HSPB9: ref=616 — RESCUED (ours=616)
  MMP12: ref=1,879 — NOT FOUND
  ZNF488: ref=4,442 — RESCUED (ours=4,442)
  RASSF5: ref=6,174 — RESCUED (ours=6,174)
  NBPF15: ref=5,306 — RESCUED (ours=5,306)
  SPON1: ref=5,639 — NOT FOUND
  RASL10B: ref=3,502 — RESCUED (ours=3,502)
  RBM15B: ref=6,600 — RESCUED (ours=6,600)
  NCOA4: ref=4,196 — RESCUED (ours=4,196)


## 5. Save Output

In [9]:
# Save gene lengths
output_path = PROCESSED_DIR / "03_gene_lengths.csv"
gene_lengths.to_csv(output_path, index=False)
print(f"Saved {len(gene_lengths):,} gene lengths to {output_path}")

# Also save the full Hubstenberger dataset with gene lengths attached
# (used by notebook 04)
hub_with_len_path = PROCESSED_DIR / "03_hub_with_gene_lengths.csv"
hub_with_len.to_csv(hub_with_len_path, index=False)
print(f"Saved {len(hub_with_len):,} genes with lengths to {hub_with_len_path}")

Saved 20,345 gene lengths to ../data/processed/03_gene_lengths.csv
Saved 15,281 genes with lengths to ../data/processed/03_hub_with_gene_lengths.csv
